<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - PWN, or Binary Exploitation</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 2: Shellcoding
</h1>

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">1 - What <em>are</em> instructions?</h1>

As we saw last time, every piece of data stored within a binary is represented as a number. For today, we will be focusing on how the code that is executed, specifically x86-64 assembly, and how we can run our own code within an (insecure) binary.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.1 - Execution of a program (again)</h2>

Looking at the same example of execution from last time, we can see that `EIP` (the instruction pointer) contains a pointer that updates as it moves down the instructions to be executed. If we were to dereference this pointer (i.e. go to the location of the address in `EIP`) we would find a whole bunch of seemingly nonsense bytes.

*Example execution of a 32-bit program:*<br>
<img src="./media/execution_example.gif">

The bytes stored at the address held in `EIP` are decoded and interpreted as instructions. If we control what these bytes are, we control what instructions are executed. If we control what instructions are executed, we can do whatever we want!

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.2 - What's our goal?</h2>

For shellcoding challenges (or more broadly, attempting to shellcode in a binary) the objective is twofold:<br>
<ol>1. Write the byte form of our instructions (i.e. shellcode) <strong>somewhere executable</strong> in memory (we'll talk about the executable part in a moment)</ol>
<ol>2. Control the instruction pointer in order to get it to point to our shellcode</ol>

Before we explain what the typical shellcode payload looks like, let's first touch on memory segment permissions.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - Memory</h1>

When a program is run, data from the file and other libraries of code it is linked are loaded into memory into different `segments` or `regions` in a `memory map`. (most) Computers do not have enough hardware in them for each individual process to have access to their own individual physical memory, so modern OSs use a trick called `virtual memory` to get around this.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.1 Virtual memory</h2>

`Virtual memory` works by assigning an `address space`, a range of values, to any program running. These values represent (i.e. can be translated to) all of the physical hardware addresses which allows each program to think (and therefore operate as though) it has access to the hardware all the time (they don't, the CPU switches between what program has access to the hardware frequently to simulate multiple processes running at once). 

Below is an example of the regions of memory within a program:

<em>Output of the `vmmap` command on a regular program in GDB:<em>

<img src="./media/example_vmmap.png">

Now let's take a look at a vulnerable program:

<em>Output of the `vmmap` command on a vulnerable program in GDB:<em>

<img src="./media/vulnerable_example_vmmap.png">

See anything different?

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.2 Memory permissions</h2>

We can see different files are mapped to different regions of memory. Within these regions, different ranges of virtual values are allocated for different sections of data within files. Files' sections may differ depending on their format but common ones include `.text` which includes executable code and `.data` which includes initialized static/global variables.

While the sections themselves are not labeled, we can make educated guesses as to what they are/contain based on the permissions of each section. Notice how the red sections (labeled `code` in the legend) have the `x` permission set under the `Perm` column. Having that `x` indicates that bytes within that section of memory are allowed to be interpreted and run as instructions. This should make it a little more clear as to what we want to do: write shellcode into an executable region of memory and gain control of `RIP` (the 64-bit equivalent to `EIP`) to point it at our code.

For now, assume that the `stack` (the purple region of memory) will always have executable permissions (i.e. if you write shellcode via a BOF you can execute it).

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">3 - Typical payloads</h1>

We now know what instructions are and where they can be executed, but how do we write our own? Even beyond that, what would we do if we had control of execution?

The typical hacker goal is to obtain a "shell" i.e. open an instance of a terminal such that you can interact with whatever system was running your binary. The two main paths for this are:
<ol>system("/bin/sh")</ol>
<ol>execve("/bin/sh", 0, 0)</ol>

`system` is a little less common (as the binary needs to contain the function `system`). If the binary isn't compiled with this function included, you won't be able to call it.

`execve` is always possible (on Linux) as it is included at the kernel level. It is called via `syscall`, a type of instruction that switches to the kernel in order to have appropriate permissions to carry out a specified operation. In the case of `execve`, it executes the program specified in the first argument, replacing the process that called it with the new process (e.g. if process1 calls `execve("process2", 0, 0)`, process1 is replaced with process2).

The first argument must contain the path to the binary you want to execute; in our case, this is typically `/bin/sh`. This will open an instance of terminal access, replacing the current program we were using with a shell. The next two arguments represent pointers to `argv` and `envp` respectively. If you are curious you can look up what they do, but for our purposes, they should be 0 (that way `execve` recognizes them as NULL and doesn't act funny with them).

As a refresher from RE, the first three arguments to any function are stored in the registers `RDI`, `RSI`, and `RDX` respectively. Thus, `RDI` must be a pointer (NOT the string itself) to the string `/bin/sh` and the other two must be 0.

One more stipulation is that the `syscall` instruction relies on the value of `RAX` to execute a particular system call. For example, if `RAX` contains `0x0`, it will call `read`. If it contains `0x5A`, it will call `chmod`. In order to call `execve`, we need to set it to `0x3B`.

Below is an example of `x64` assembly that sets this up:

```c
mov rdi, 0x7ffff3e659 // Example pointer to /bin/sh
mov rsi, 0
mov rdx, 0
mov rax, 0
syscall
```

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Practice: Byte Parser</h1>

This program will take in whatever bytes you input and execute them as code. Try and get a shell!

In [1]:
!mkdir -p /home/user/introsec/pwn/shellcoding/binaries

In [6]:
%cd ~/introsec/pwn/shellcoding/binaries

/home/user/introsec/pwn/shellcoding/binaries


In [3]:
%%file byteparser.c
#include <stdio.h>
#include <stdlib.h>

//gcc byteparser.c -o byteparser -fno-stack-protector -no-pie -z execstack

int main(void){

    char code[48];
    printf("Enter the bytes of your shellcode here: ");
    fgets(code, sizeof(code), stdin);

    ((void(*)())code)(); // This will call your shellcode automatically! Don't worry about overwriting the return address
    return 0;
}

Overwriting byteparser.c


In [4]:
!gcc byteparser.c -o byteparser -fno-stack-protector -no-pie -z execstack

<h3>IMPORTANT!</h3>

Run your solve file outside of the notebook since `p.interactive()` won't work properly if you use it inside of here.

In [5]:
%%file parsesolve.py
from pwn import *

context.update(arch='amd64', os='linux')
p = process("./byteparser")

shellcode = asm('''
# Your assembly here!
''')

p.interactive()

Overwriting parsesolve.py


<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">4 - NULL Bytes</h1>

An issue you may come across is that NULL bytes (`0x00`) cause certain data parsing functions to stop receiving input. This is because, in C, strings are NULL terminated. That is, the end of a string is denoted by a null byte. Let's take a look at an example instruction that includes a null byte:

```c
mov rsi, 0
```

In byte form: `0x48 c7 c6 00 00 00 00`

Notice the multiple `0x00` at the end of the instruction string. Some functions may stop at those null bytes and will not copy data from further on. Here's an example program that does just that:

In [6]:
%%file nullbyte.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

//gcc nullbyte.c -o nullbyte

int main(void){

    char first[12];
    char second[12];
    memset(first, 0, sizeof(first));
    memset(second, 0, sizeof(second));

    printf("Enter your bytes here:\n");
    fgets(first, sizeof(first), stdin);
    strcpy(second, first);

    printf("Here's what is stored in the first buffer (after the fgets()): ");
    for (int i = 0; i < sizeof(first); i++)
        printf("%02x ", first[i]);
    printf("\n");

    printf("Here's what is stored in the second buffer (after the strcpy()): ");
    for (int i = 0; i < sizeof(second); i++)
        printf("%02x ", second[i]);
    printf("\n");

    return 0;
}

Overwriting nullbyte.c


In [7]:
!gcc nullbyte.c -o nullbyte

In [8]:
%%file testnull.py
from pwn import *

context.update(arch='amd64', os='linux')
p = process("./nullbyte")

bytestring = b'\x01\x02\x03\x04\x05\x00\x07\x08\x09\x10'

print(p.readline())
p.sendline(bytestring)
print(p.readline()[:-2])
print(p.readline()[:-2])

Overwriting testnull.py


In [9]:
!python3 testnull.py

[x] Starting local process './nullbyte'
[<] 
Starting local process './nullbyte': pid 6215
[+] 
b'Enter your bytes here:\n'
b"Here's what is stored in the first buffer (after the fgets()): 01 02 03 04 05 00 07 08 09 10 0a 00"
b"Here's what is stored in the second buffer (after the strcpy()): 01 02 03 04 05 00 00 00 00 00 00 00"
[*] Process './nullbyte' stopped with exit code 0 (pid 6215)


As you can see, the `strcpy()` in used to move data from the first to second buffer does not copy the bytes after the `0x00` in the middle. Let's try changing it to another value and examining the results.

In [10]:
%%file testnull.py
from pwn import *

context.update(arch='amd64', os='linux')
p = process("./nullbyte")

bytestring = b'\x01\x02\x03\x04\x05\x06\x07\x08\x09\x10' # Changed 0x00 -> 0x06

print(p.readline())
p.sendline(bytestring)
print(p.readline()[:-2])
print(p.readline()[:-2])

Overwriting testnull.py


In [11]:
!python3 testnull.py

[x] Starting local process './nullbyte'
[q] 
Starting local process './nullbyte': pid 44805
[+] 
b'Enter your bytes here:\n'
b"Here's what is stored in the first buffer (after the fgets()): 01 02 03 04 05 06 07 08 09 10 0a 00"
b"Here's what is stored in the second buffer (after the strcpy()): 01 02 03 04 05 06 07 08 09 10 0a 00"
[*] Process './nullbyte' stopped with exit code 0 (pid 44805)


Now they are the same! This is important to keep in mind as some instructions may have null bytes in them. For example, as shown above, `mov`ing 0 into a register will include `0x00` in the instruction. Not only `0x00`, but other bytes might be banned since they commonly show up in shellcode payloads. So how can we get around this?

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">4.1 - NOP Sled</h2>

A common situation with shellcoding (or memory corruption in general) is that you won't know *precisely* where your code will be in memory. I'm not talking about ASLR (yet), just that you might not know the exact stack address you are writing to without significant trial and error (and debugging). An easy way to avoid this is to use a `NOP` sled. What does that look like?

In assembly:

```c
nop // NOTE: The byte representation of a NOP is 0x90
nop
nop
nop
nop
nop
nop
nop
nop
nop
nop
nop
nop
nop
nop
mov rdi, 0x7ffff3e659 // Example pointer to /bin/sh
mov rsi, 0
mov rdx, 0
mov rax, 0
syscall
```

While this might look silly, it can be very effective! Rather than spending all the required time to figure out what precise address you are writing to is (and dealing with alignment quirks), you can simply have `RIP` hit one of your many `NOP`s and it will slide down to your shellcode (hence the name)!

Speaking of alignment, I'll throw in a bonus technique. Many architectures require you to be byte-aligned, that is, to have all instruction lengths as a multiple of some number (and therefore the addresses at which they execute will also be consistent). For example, if you have an architecture with instructions that are 2-byte aligned, your execution might look like this:

```c
0x0: instr1
0x2: instr2
0x4: instr3 //Length 4
0x8: instr4
0xa: instr5 //Length 4
0xe: instr6
0x10: instr8 //Length 8
0x18: instr9 //Length 6
0x20: instr10:
```

Notice that each instruction has a length that is some multiple of 2. What if your shellcode ends up on address `0x3`? You need to align it!

This is easily solved by sending in a few `NOP`s. Following along with the arbitrary address of `0x3`, you can send in one `NOP` beforehand and force your first real instruction to be on an aligned address (i.e. `0x4`) without having to use NULL bytes!

In [3]:
%%file noptime.c
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <time.h>
#include <string.h>

#define ARRAY_SIZE 1024  // Size of the NOP sled array

// gcc noptime.c -o noptime -fno-stack-protector -no-pie -z execstack -Wno-all

void execute_shellcode(uint8_t *array, size_t offset) {
  if (array[offset] == 0) {
    printf("Nothing detected at offset %zu. Exiting...\n", offset);
    exit(0);
  }

  printf("Executing shellcode at offset %zu...\n", offset);
  ((void (*)())(array + offset))();
}

int main() {
  uint8_t array[ARRAY_SIZE] = {0};  // Initialize array with NOPs (zeros)
  size_t random_offset;

  // Seed random number generator
  srand(time(NULL));

  printf("Welcome to the challenge!\n");

  // Get user input for shellcode
  printf("Enter your input:\n> ");
  char input[ARRAY_SIZE];  // Match size of the array
  fgets(input, sizeof(input), stdin);

  // Remove newline character from input
  size_t input_length = strcspn(input, "\n");
  input[input_length] = '\0';

  // Copy input into the array
  strncpy((char *)array, input, ARRAY_SIZE - 1);
  array[ARRAY_SIZE - 1] = '\0';

  printf("Checking array.\n");

  // Pick a random offset in the array
  random_offset = rand() % ARRAY_SIZE;

  // Execute shellcode at the random offset
  execute_shellcode(array, random_offset);

  return 0;
}

Overwriting noptime.c


In [4]:
!gcc noptime.c -o noptime -fno-stack-protector -no-pie -z execstack -Wno-all

In [5]:
%%file nopsolve.py
from pwn import *

context.update(arch='amd64', os='linux')
p = process("./noptime")

shellcode = asm('''
Your assembly here!''')

p.sendline(shellcode)

Writing nopsolve.py


<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">4.2 - In conclusion...</h2>

Creativity is your friend here. Consider your goal and think of alternative ways to get the value you need into a register. If a certain value is banned (e.g. you can't send in the byte `0x3b`), can you send in multiple values and use `add` to sum up to it? If you need to zero out a register, think about instructions (there are two in particular!) that will get the register's value to 0 without actually loading in the value 0.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Putting it all together: Code of Your Own</h1>

In [6]:
%%file codeofyourown.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <sys/mman.h>

// Function to validate shellcode for disallowed instructions (e.g., "mov")
int validate_shellcode(const char *shellcode, int size) {
  
  
//   Opcodes for `mov`
  const char disallowed_opcodes[] = {
    0x89, 0x8B,          // General-purpose `mov` instructions
    0xB8, 0xB9, 0xBA, 0xBB, 0xBC, 0xBD, 0xBE, 0xBF // `mov` with immediate values
  };

  // Opcodes for `push or pop`
//   const char disallowed_opcodes[] = {
//     // PUSH opcodes
//     0x50, 0x51, 0x52, 0x53, 0x54, 0x55, 0x56, 0x57, // `push` rax to rdi
//     0x68,                                          // `push imm32`
//     0x6A,                                          // `push imm8`

//     // POP opcodes
//     0x58, 0x59, 0x5A, 0x5B, 0x5C, 0x5D, 0x5E, 0x5F  // `pop` rax to rdi
//   };
  
//   const char disallowed_opcodes[] = {
//     'b', 'i', 'n', 's', 'h', '/'
//   };


  int disallowed_count = sizeof(disallowed_opcodes) / sizeof(disallowed_opcodes[0]);

  // look for for disallowed opcodes
  for (int i = 0; i < size; i++) {
    for (int j = 0; j < disallowed_count; j++) {
      if (shellcode[i] == disallowed_opcodes[j]) {
        printf("Invalid shellcode: instruction detected at byte %u (0x%02X)\n", i, shellcode[i]);
        return 0; // Invalid shellcode
      }
    }
  }

  return 1; // Valid shellcode
}

void vuln() {
  char buffer[128];

  printf("Give your input:\n> ");
  fgets(buffer, sizeof(buffer), stdin);

  // Validate the shellcode
  if (!validate_shellcode(buffer, sizeof(buffer))) {
    printf("Shellcode validation failed. Execution aborted.\n");
    return;
  }

  printf("Shellcode is valid. Executing...\n");
  ((void (*)())buffer)();
}

int main() {
  setvbuf(stdout, NULL, _IONBF, 0);
  printf("Welcome to the challenge!\n");
  vuln();
  return 0;
}

Writing codeofyourown.c


In [7]:
!gcc codeofyourown.c -o codeofyourown -fno-stack-protector -no-pie -z execstack -Wno-all

In [8]:
%%file codesolve.py
from pwn import *

context.update(arch='amd64', os='linux')
p = process("./codeofyourown")

shellcode = asm('''
Your assembly here!''')

p.sendline(shellcode)

Writing codesolve.py
